# propygator — a gentle introduction

`propygator` is a small library for **orbital simulation**: you give it where a
satellite is and how fast it's going, and it tells you where it will be over time —
then helps you plot and export the result.

This is the five-minute on-ramp. For the step-by-step tour see
`02_numerical_propagation.ipynb`; for a quick multi-orbit showcase see `03_demo.ipynb`.

**Before you run this:** propygator needs the ~500 MB *orekit-data* bundle (physical
reference data — leap seconds, Earth orientation, ephemerides, space weather). The most
reliable setup is to keep one copy at `~/.propygator/orekit-data/` — see
`docs/orekit_setup_reference.md`. The first propagation also starts a Java VM under the
hood (a few seconds); that's normal.

In [ ]:
import numpy as np

import propygator as pgr

# Importing does NOT start the Java VM — the first propagation does, lazily.
pgr.__version__

## 1 · Describe where the satellite is

A **`State`** is the starting point: a position and a velocity, at a moment in time
(`Epoch`), in a reference frame (`Frame`). Everything is SI — metres and metres per
second. Here we set up a circular orbit ~500 km above the Earth.

In [ ]:
MU = 3.986004418e14  # Earth's gravity parameter, m^3/s^2
r = 6_378_137.0 + 500_000.0  # distance from Earth's centre (radius + 500 km)
speed = np.sqrt(MU / r)  # the speed that gives a circular orbit at this radius

state = pgr.State(
    pgr.Epoch.from_iso("2024-01-01T00:00:00"),  # time (UTC)
    np.array([r, 0.0, 0.0]),  # position (x, y, z), metres
    np.array([0.0, speed, 0.0]),  # velocity (vx, vy, vz), m/s
    pgr.Frame.EME2000,  # a standard inertial frame
)
state

## 2 · Propagate it forward

`propagate_numerical` integrates the orbit forward under a realistic force model (Earth
gravity, Sun and Moon, atmospheric drag, solar radiation pressure). You give it a
`duration` and an `output_step`, both in seconds, and it returns a **`Trajectory`** —
the orbit sampled over time.

In [ ]:
traj = pgr.propagate_numerical(state, duration=6000, output_step=60)  # ~one orbit
print(len(traj), "samples, in frame", traj.frame)

## 3 · Look at the result

A `Trajectory` behaves like a sequence of `State`s and can report familiar orbital
("Keplerian") elements — semi-major axis, eccentricity, inclination.

In [ ]:
kep = traj[0].to_keplerian()
print(f"altitude    : {(kep.semi_major_axis_m - 6_378_137.0) / 1000:.0f} km")
print(f"eccentricity: {kep.eccentricity:.4f}   (0 = perfectly circular)")
print(f"inclination : {np.degrees(kep.inclination_rad):.1f} deg")

## 4 · Plot it

Each `plot_*` returns a normal matplotlib or Plotly figure. `plot_summary` stacks the
ground track, altitude, and speed into one figure; `plot_3d` is an interactive 3-D view
you can rotate.

In [ ]:
pgr.plot_summary(traj);

In [ ]:
pgr.plot_3d(traj).show()

---

That's the whole idea: **`State` → `propagate_numerical` → `Trajectory` → plot**.

Next:
- `02_numerical_propagation.ipynb` — force models, spacecraft, attitude, variable drag,
  the altitude guards, and exports.
- `03_demo.ipynb` — four contrasting orbits, end to end.